# QLoRA Fine-Tuning — Primary Model (Colab)

**AAI-590 Capstone** · Team: Evin Joy, Sabina George, Jagadeesh Kumar Sellapan

Fine-tunes a compact open-weight LLM for medical question answering using
**QLoRA** (4-bit quantised base + trainable low-rank adapters). This is the
project's *primary* model; the from-scratch Transformer is the baseline.

**Requirements**
- A **GPU runtime** (Colab: *Runtime → Change runtime type → T4 GPU* or better).
- If the base model is gated (Mistral / Llama), a Hugging Face access token with
  the model's license accepted.

**Pipeline:** install deps → load & format MedMCQA + MedQA → load base model in
4-bit + LoRA → SFT training → accuracy check → save adapter.

### 1&nbsp;&nbsp;Install dependencies

In [ ]:
# Colab GPU runtime required:  Runtime > Change runtime type > T4 GPU (or better)
# Install only what Colab lacks. Do NOT pin old versions of transformers/datasets
# — that downgrades numpy and conflicts with Colab's preinstalled stack. Colab
# already ships recent, numpy-2-compatible transformers, datasets, and torch.
!pip install -q -U peft bitsandbytes accelerate

### 2&nbsp;&nbsp;Imports and GPU check

In [ ]:
import os, re, random
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, TrainingArguments)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

assert torch.cuda.is_available(), \
    "No GPU detected. In Colab: Runtime > Change runtime type > GPU."
print("GPU:", torch.cuda.get_device_name(0))
random.seed(42); torch.manual_seed(42)

### 3&nbsp;&nbsp;Configuration

In [ ]:
# --- Configuration --------------------------------------------------------- #
CFG = dict(
    # Primary model: a compact open-weight LLM. Mistral is *gated* on the Hub,
    # so accept its license and set an access token (next cell), OR switch to an
    # ungated model such as "Qwen/Qwen2.5-7B-Instruct" by editing model_name.
    model_name   = "mistralai/Mistral-7B-Instruct-v0.2",
    n_train      = 8000,     # training questions (raise once the pipeline works)
    n_val        = 500,      # held-out questions for the accuracy check
    max_seq_len  = 768,      # covers question + 4 options comfortably (see EDA)
    # QLoRA / LoRA adapter settings
    lora_r       = 16,
    lora_alpha   = 32,
    lora_dropout = 0.05,
    # Optimisation
    learning_rate = 2e-4,
    epochs        = 1,
    batch_size    = 2,       # per-device; effective batch = batch_size * grad_accum
    grad_accum    = 8,
    output_dir    = "qlora-medqa-adapter",
)

### 4&nbsp;&nbsp;Hugging Face access token (for gated models)

In [ ]:
# If the chosen base model is gated (Mistral, Llama, ...), authenticate once.
# Get a token at https://huggingface.co/settings/tokens and paste it below,
# or run `from huggingface_hub import login; login()` and enter it interactively.
# os.environ["HF_TOKEN"] = "hf_xxx"   # <-- uncomment and set, or use login()

### 5&nbsp;&nbsp;Data loading and instruction formatting

In [ ]:
# --- Load and format the medical MCQ data --------------------------------- #
# We pull MedMCQA and MedQA directly from the Hub and format each item as an
# instruction. The option order is shuffled (and the gold label remapped) to
# neutralise the positional bias identified during exploratory analysis.
LETTERS = "ABCD"

PROMPT = (
    "You are a medical expert answering a multiple-choice question. "
    "Select the single best option.\n\n"
    "Question: {q}\n"
    "A. {a}\nB. {b}\nC. {c}\nD. {d}\n\n"
    "Answer:"
)

def _records(n):
    """Yield up to n unified {question, options, answer_idx} records."""
    out = []
    medmcqa = load_dataset("openlifescienceai/medmcqa", split="train")
    for r in medmcqa:
        opts = [r["opa"], r["opb"], r["opc"], r["opd"]]
        if all(o for o in opts) and r["cop"] in (0, 1, 2, 3):
            out.append({"question": r["question"].strip(), "options": opts,
                        "answer_idx": int(r["cop"])})
        if len(out) >= n:
            return out
    medqa = load_dataset("GBaker/MedQA-USMLE-4-options", split="train")
    for r in medqa:
        o = r["options"]
        opts = [o["A"], o["B"], o["C"], o["D"]]
        idx = "ABCD".index(r["answer_idx"])
        out.append({"question": r["question"].strip(), "options": opts, "answer_idx": idx})
        if len(out) >= n:
            break
    return out

def _format(rec, with_answer=True, shuffle=True):
    opts, idx = list(rec["options"]), rec["answer_idx"]
    if shuffle:
        order = list(range(4)); random.shuffle(order)
        opts = [rec["options"][i] for i in order]
        idx = order.index(rec["answer_idx"])
    text = PROMPT.format(q=rec["question"], a=opts[0], b=opts[1], c=opts[2], d=opts[3])
    if with_answer:
        text = f"{text} {LETTERS[idx]}. {opts[idx]}"
    return text, idx

def build_datasets(cfg, eos_token):
    recs = _records(cfg["n_train"] + cfg["n_val"])
    random.shuffle(recs)
    train_recs, val_recs = recs[:cfg["n_train"]], recs[cfg["n_train"]:cfg["n_train"] + cfg["n_val"]]
    # Training text ends with the answer and an EOS token so the model learns to stop.
    train_texts = [_format(r, with_answer=True)[0] + eos_token for r in train_recs]
    train_ds = Dataset.from_dict({"text": train_texts})
    return train_ds, val_recs

### 6&nbsp;&nbsp;Load the base model in 4-bit and attach LoRA

In [ ]:
# --- Load the base model in 4-bit and attach LoRA adapters ---------------- #
# QLoRA: the pretrained weights are quantised to 4-bit NormalFloat and frozen;
# only the injected low-rank adapters are trained.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,   # use torch.bfloat16 on A100/L4
)

tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    CFG["model_name"], quantization_config=bnb_config,
    device_map="auto", torch_dtype=torch.float16,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=CFG["lora_dropout"],
    bias="none", task_type="CAUSAL_LM",
    # Adapt all attention and MLP projections (standard for Llama/Mistral/Qwen).
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

### 7&nbsp;&nbsp;Build the training set

In [ ]:
train_ds, val_recs = build_datasets(CFG, tokenizer.eos_token)
print(f"Train examples: {len(train_ds):,} | Val examples: {len(val_recs):,}")
print("\n--- Example training text ---\n")
print(train_ds[0]["text"][:600])

### 8&nbsp;&nbsp;Fine-tune

In [ ]:
# --- Supervised fine-tuning with the (stable) transformers Trainer -------- #
from transformers import Trainer, DataCollatorForLanguageModeling

# Tokenise the formatted text; the collator builds causal-LM labels and pads.
def tokenize_fn(ex):
    return tokenizer(ex["text"], truncation=True, max_length=CFG["max_seq_len"])

tokenized = train_ds.map(tokenize_fn, remove_columns=["text"])
collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

train_args = TrainingArguments(
    output_dir=CFG["output_dir"],
    per_device_train_batch_size=CFG["batch_size"],
    gradient_accumulation_steps=CFG["grad_accum"],
    learning_rate=CFG["learning_rate"],
    num_train_epochs=CFG["epochs"],
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,                       # bf16=True on A100/L4 instead
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
)
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenized,
    data_collator=collator,
)
trainer.train()

### 9&nbsp;&nbsp;Evaluate (held-out accuracy)

In [ ]:
# --- Quick accuracy check on held-out questions --------------------------- #
# Greedy-decode a single answer letter per question and compare to the gold key.
model.eval()

@torch.no_grad()
def predict_letter(rec):
    prompt, gold_idx = _format(rec, with_answer=False, shuffle=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=4, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    gen = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    m = re.search(r"[ABCD]", gen.upper())
    pred_idx = "ABCD".index(m.group(0)) if m else 0
    return pred_idx, gold_idx

correct = sum(int(p == g) for p, g in (predict_letter(r) for r in val_recs))
acc = correct / len(val_recs)
print(f"Fine-tuned validation accuracy: {acc:.3f} on {len(val_recs)} questions "
      f"(random chance = 0.25)")

### 10&nbsp;&nbsp;Save the adapter

In [ ]:
# --- Save the trained adapter --------------------------------------------- #
# Only the small LoRA adapter is saved (a few tens of MB), not the full base model.
model.save_pretrained(CFG["output_dir"])
tokenizer.save_pretrained(CFG["output_dir"])
print("Saved adapter to:", CFG["output_dir"])

# To download from Colab:
# import shutil; shutil.make_archive(CFG["output_dir"], "zip", CFG["output_dir"])
# from google.colab import files; files.download(CFG["output_dir"] + ".zip")

# To reload later for inference:
#   from peft import PeftModel
#   base = AutoModelForCausalLM.from_pretrained(CFG["model_name"], quantization_config=bnb_config, device_map="auto")
#   model = PeftModel.from_pretrained(base, CFG["output_dir"])

### Notes

- **Out-of-memory?** Lower `max_seq_len` (e.g. 512), keep `batch_size=1` and raise
  `grad_accum`, or reduce `n_train`. Gradient checkpointing and the paged optimiser
  are already enabled.
- **Faster GPU (A100/L4):** set `bnb_4bit_compute_dtype=torch.bfloat16` and
  `bf16=True` (instead of `fp16=True`).
- **Baseline comparison:** run cell 9 *before* training (on the base model) to get
  the zero-shot accuracy, then again after training — the difference is the effect
  of fine-tuning, reported in the Results section.

*Use of AI tools:* a generative AI assistant helped scaffold and comment this
notebook; the team configures, runs, and verifies the training on GPU hardware.